<a href="https://colab.research.google.com/github/saketchoudhary2147-hue/HMM-project/blob/main/HMM_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
import os
import math
import numpy as np

states = { "s": 0, "E": 1, "5": 2, "I" : 3, "e": 4}
id2state = {0: "s", 1: "E", 2: "5", 3: "I", 4: "e"}

state_transition_prob = np.array([[0.0, 1.0, 0.0, 0.0, 0.0],
                                  [0.0, 0.9, 0.1, 0.0, 0.0],
                                  [0.0, 0.0, 0.0, 1.0, 0.0],
                                  [0.0, 0.0, 0.0, 0.9, 0.1],
                                  [0.0, 0.0, 0.0, 0.0, 0.0]])
emission_nuc_codes = {'A': 0,
                      'C': 1,
                      'G': 2,
                      'T': 3}

emission_probs = np.array([[0.00, 0.00, 0.00, 0.00],
                           [0.25, 0.25, 0.25, 0.25],
                           [0.05, 0.00, 0.95, 0.00],
                           [0.40, 0.10, 0.10, 0.40],
                           [0.00, 0.00, 0.00, 0.00]])

query_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"


In [ ]:
def get_log_prob_for_state_path (state_path, query_sequence):
    res = math.log(0.25)
    for i in range(1, len(state_path)):
        res += math.log(state_transition_prob[ states[state_path[i-1]] ][ states[state_path[i]] ]*emission_probs[ states[state_path[i]] ][ emission_nuc_codes[query_sequence[i]] ])
    return res

In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEE5IIIIIIIIIIIIIIIIIII
k1 = get_log_prob_for_state_path("EEEEEE5IIIIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") +  math.log (0.1)
print (k1)


-43.89740030179307


# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEE5IIIIIIIIIIIIIIIIIII
k1 = get_log_prob_for_state_path("EEEEEE5IIIIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") +  math.log (0.1)
print (k1)


In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEE5IIIIIIIIIIIIIIIII
k2 = get_log_prob_for_state_path("EEEEEEEE5IIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k2)


-43.45111319916465


In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEE5IIIIIIIIIIIII
k3 = get_log_prob_for_state_path("EEEEEEEEEEEE5IIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k3)


-43.944833355027704


In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEE5IIIIIIIIII
k4 = get_log_prob_for_state_path("EEEEEEEEEEEEEEE5IIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k4)

-42.58225552052512


In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEEEEE5IIIIIII
k5 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k5)


-41.21967768602254


In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEEEEEEEEE5III
k6 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEE5III", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k6)


-41.713397841885595


In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEEEEEEEEEEEEE
only_E = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEEEEEE", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (only_E)

-40.98025137355685


### Design of the Viterbi Value matrix

Rows correspond to the hidden states, and the columns correspond to the emissions that is the observed nucleotide sequences. Here I am showing the calculation for the first two nucletides.

```
             C                                                          T     T
s [s-s-C(0.00) max(s-s-C-s-T, s-E-C-s-T, s-5-C-s-T, s-I-C-s-T, s-e-C-s-T)     .]
E [s-E-C(0.25) max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)     .]
5 [s-5-C(0.00) max(s-s-C-5-T, s-E-C-5-T, s-5-C-5-T, s-I-C-5-T, s-e-C-5-T)     .]
I [s-I-C(0.00) max(s-s-C-I-T, s-E-C-I-T, s-5-C-I-T, s-I-C-I-T  s-e-C-I-T)     .]
e [s-e-C(0.00) max(s-s-C-e-T, s-E-C-e-T, s-5-C-e-T, s-I-C-e-T, s-e-C-e-T)     .]

```

It is important to remember that you will be working in the log scale.

In [ ]:
import numpy as np
import math

# 1. Initialize dimensions
n_states = len(states)
n_obs = len(query_sequence)

# 2. Initialize matrices
viterbi_value_matrix = np.full((n_states, n_obs), -np.inf)
viterbi_trace_matrix = np.full((n_states, n_obs), -1, dtype=int)

# 3. Handle first column (initial step)
for state_idx in range(n_states):
    trans_prob = state_transition_prob[states['s'], state_idx]
    if trans_prob > 0:
        nuc_idx = emission_nuc_codes[query_sequence[0]]
        emiss_prob = emission_probs[state_idx, nuc_idx]
        if emiss_prob > 0:
            viterbi_value_matrix[state_idx, 0] = math.log(trans_prob) + math.log(emiss_prob)

print(f"Matrices initialized for sequence length {n_obs} and {n_states} states.")

Matrices initialized for sequence length 26 and 5 states.


### Implementation of Viterbi Algorithm
Write a function `calculate_prob_for_a_node()` that populate a single cell in the matrix. The function will return two values:
1. the maximum value, for example, look at the 2nd row, 2nd column in the matrix: `max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)`. If the probability for `s-E-C-E-T` is highest (lets say X), then the function should return `X`

**AND**

2. The index of that maximum value described in the first point: so index of X is `1` (recall that Python works on the 0-based index system)

- Populate `viterbi_value_matrix` with `X` for row 2 and col 2

- Populate `viterbi_trace_matrix` with `1` for row 2 and col 2

In [ ]:
def calculate_prob_for_a_node(current_state_idx, time_step):
    nuc_idx = emission_nuc_codes[query_sequence[time_step]]
    emiss_prob = emission_probs[current_state_idx, nuc_idx]

    if emiss_prob == 0:
        return -np.inf, -1

    log_emiss = math.log(emiss_prob)
    best_prob = -np.inf
    best_prev_state = -1

    for prev_state_idx in range(n_states):
        trans_prob = state_transition_prob[prev_state_idx, current_state_idx]
        if trans_prob > 0:
            prev_val = viterbi_value_matrix[prev_state_idx, time_step - 1]
            if prev_val != -np.inf:
                prob = prev_val + math.log(trans_prob) + log_emiss
                if prob > best_prob:
                    best_prob = prob
                    best_prev_state = prev_state_idx

    return best_prob, best_prev_state

# Execute Viterbi Loop
for t in range(1, n_obs):
    for s in range(n_states):
        prob, trace = calculate_prob_for_a_node(s, t)
        viterbi_value_matrix[s, t] = prob
        viterbi_trace_matrix[s, t] = trace

print("Viterbi matrix population complete.")
import pandas as pd
display(pd.DataFrame(viterbi_value_matrix, index=states.keys(), columns=list(query_sequence)))

Viterbi matrix population complete.


,C,T,T,C,A,T,G,T,G,A,...,A,C,G,T,A,A,G,T,C,A
s,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,...,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf
E,-1.386294,-2.877949,-4.369604,-5.861259,-7.352914,-8.844569,-10.336224,-11.827878,-13.319533,-14.811188,...,-25.252772,-26.744427,-28.236082,-29.727737,-31.219392,-32.711047,-34.202702,-35.694357,-37.186011,-38.677666
5,-inf,-inf,-inf,-inf,-11.159576,-inf,-11.198447,-inf,-14.181757,-18.617851,...,-29.059435,-inf,-29.098306,-inf,-35.026054,-36.517709,-35.064925,-inf,-inf,-42.484329
I,-inf,-inf,-inf,-inf,-inf,-12.075867,-14.483813,-12.114738,-14.522683,-15.098048,...,-25.539632,-27.947577,-30.355523,-30.014596,-31.036248,-32.057899,-34.465844,-35.487496,-37.895441,-38.917093
e,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,...,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf


In [ ]:
def get_viterbi_path():
    # Find the best final state at the last nucleotide
    last_col = viterbi_value_matrix[:, -1]
    best_last_state = np.argmax(last_col)
    max_log_prob = last_col[best_last_state]

    if max_log_prob == -np.inf:
        return "No valid path found.", -np.inf

    # Backtrack
    path = [best_last_state]
    for t in range(n_obs - 1, 0, -1):
        prev_state = viterbi_trace_matrix[path[-1], t]
        path.append(prev_state)

    # Reverse and map to state names
    path.reverse()
    state_path = "".join([id2state[s] for s in path])
    return state_path, max_log_prob

final_path, final_log_prob = get_viterbi_path()
print(f"Sequence:   {query_sequence}")
print(f"State Path: {final_path}")
print(f"Log Prob:   {final_log_prob}")

Sequence:   CTTCATGTGAAAGCAGACGTAAGTCA
State Path: EEEEEEEEEEEEEEEEEEEEEEEEEE
Log Prob:   -38.677666280562796
